***<h1>Transfermarket Dataset</h1>***

Guardamos en un dataset el nombre y valor de mercado unicamente de los jugadores en activo basandonos en last_season sea 2024 

In [3]:
import pandas as pd

# 1. CARGA DE DATOS
# Asegúrate de reemplazar 'ruta/al/...' con la ubicación real de tus archivos
df_players = pd.read_csv('../data/raw/players.csv')
df_valuations = pd.read_csv('../data/raw/player_valuations.csv')

# --- 2. PREPARACIÓN DE VALORACIONES (Obtener la más reciente) ---

# Convertir la columna 'date' a datetime
df_valuations['date'] = pd.to_datetime(df_valuations['date'])

# Encontrar el índice de la valoración MÁS RECIENTE para CADA jugador
idx_max_date = df_valuations.groupby('player_id')['date'].idxmax()
df_latest_valuations = df_valuations.loc[idx_max_date].copy()

# Seleccionar solo ID y valor de mercado
df_values_only = df_latest_valuations[['player_id', 'market_value_in_eur']]


# --- 3. FILTRO DE JUGADORES ACTIVOS (last_season = 2024) ---

ACTIVE_SEASON = 2024

# Filtrar la tabla de jugadores original para obtener solo los IDs de los activos
df_active_players = df_players[df_players['last_season'] == ACTIVE_SEASON].copy()
active_player_ids = df_active_players['player_id']


# --- 4. FILTRO FINAL Y FUSIÓN ---

# Filtrar las valoraciones más recientes para mantener SOLO a los jugadores activos
df_final_filtered = df_values_only[
    df_values_only['player_id'].isin(active_player_ids)
].copy()

# Preparar la tabla de nombres
df_names = df_players[['player_id', 'name']]

# Fusionar con la tabla de nombres
df_output = pd.merge(
    df_final_filtered,
    df_names,
    on='player_id',
    how='left'
)

# 5. RESULTADO FINAL (Seleccionando solo las columnas requeridas)
df_final_dataset = df_output[['name', 'market_value_in_eur']]

print(f"Número de jugadores activos con valoración más reciente: {len(df_final_dataset)}")
print("Primeras 5 filas del dataset final:")
print(df_final_dataset.head())

# 6. EXPORTACIÓN DEL PRODUCTO FINAL
df_final_dataset.to_csv('.../data/raw/market_values.csv', index=False)

Número de jugadores activos con valoración más reciente: 6296
Primeras 5 filas del dataset final:
                  name  market_value_in_eur
0         James Milner              1000000
1  Anastasios Tsokanis               300000
2        Jonas Hofmann              3000000
3           Pepe Reina               600000
4        Lionel Carole               400000


OSError: Cannot save file into a non-existent directory: '...\data\raw'